In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

colors = [
    ROOT.kRed,
    ROOT.kBlue,
    ROOT.kGreen + 2,
    ROOT.kOrange + 7,
    ROOT.kMagenta,
    ROOT.kCyan + 2,
    ROOT.kViolet,
    ROOT.kAzure + 7,
    ROOT.kPink + 7,
    ROOT.kTeal + 3,
    ROOT.kSpring + 5,
    ROOT.kYellow + 2,
    ROOT.kOrange + 1,
    ROOT.kBlue - 7,
    ROOT.kRed - 7,
    ROOT.kGray + 2,
    ROOT.kBlack,
    ROOT.kSpring - 7
]

In [5]:
files = sorted((project_root/'data/261012').glob('SUBT_*.csv'))
#files = [project_root/'data/SummingChannelTest/07_SumChTest_BG.csv', project_root/'data/SummingChannelTest/08_SumChTest_137Cs.csv', 
#         project_root/'data/SummingChannelTest/09_SumChTest_22Na.csv', project_root/'data/SummingChannelTest/10_SumChTest_60Co.csv']

#files = [project_root/'data/261009/02_18ChTest.csv']


acqs = [SiphraAcquisition(file) for file in files]

#file = project_root/'data/SummingChannelTest/SUBT_10_SumChTest_60Co.csv'
#file_bg = project_root/'data/SummingChannelTest/SUBT_06_SumChTest_BG.csv'
#file = project_root/'data/NewBoardsTest/01_NewBoardTest.csv'

#acq_src = SiphraAcquisition(file)
#acq_bg = SiphraAcquisition(file_bg)

#acqs = [acq_bg, acq_src]

print(acqs)

[SIPHRAAcquisition(File: 'SUBT_A_01_18ChTest_BG'), SIPHRAAcquisition(File: 'SUBT_B_01_18ChTest_BG')]


In [6]:
#histogram
hists = {}
#acq = acq_src

#sources = ['BG', 'Cs-137', 'Na-22', 'Co-60']
sources = ['BG_a', 'BG_b']

channels = ['Ch1', 'Ch2', "Ch3", "Ch4", "Ch5", "Ch6", "Ch7", "Ch8", "Ch9", "Ch10", "Ch11", "Ch12", "Ch13", "Ch14",
            "Ch15", "Ch16", "Siphra_sum", "Manual_sum"]
for i, source in enumerate(sources):
    
    acq = acqs[i]
    for channel in channels:
        hist = ROOT.TH1F(f"{channel}_{source}", "", BITS12, 0, BITS12)
        if channel == "Manual_sum":
            hist.Fill(acq[channel]/16)
        else: 
            hist.Fill(acq[channel])
    
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle(r"Counts")
        hists[f"{channel}_{source}"] = hist
        del hist
print(hists)

{'Ch1_BG_a': <cppyy.gbl.TH1F object at 0x60fbc7e99dd0>, 'Ch2_BG_a': <cppyy.gbl.TH1F object at 0x60fbc3843e30>, 'Ch3_BG_a': <cppyy.gbl.TH1F object at 0x60fbc7e641f0>, 'Ch4_BG_a': <cppyy.gbl.TH1F object at 0x60fbc75d1da0>, 'Ch5_BG_a': <cppyy.gbl.TH1F object at 0x60fbc75c8bc0>, 'Ch6_BG_a': <cppyy.gbl.TH1F object at 0x60fbc73d9750>, 'Ch7_BG_a': <cppyy.gbl.TH1F object at 0x60fbc39bf120>, 'Ch8_BG_a': <cppyy.gbl.TH1F object at 0x60fbc39e0060>, 'Ch9_BG_a': <cppyy.gbl.TH1F object at 0x60fbc3642e00>, 'Ch10_BG_a': <cppyy.gbl.TH1F object at 0x60fbc2c45170>, 'Ch11_BG_a': <cppyy.gbl.TH1F object at 0x60fbc2dcb9f0>, 'Ch12_BG_a': <cppyy.gbl.TH1F object at 0x60fbc2e14d00>, 'Ch13_BG_a': <cppyy.gbl.TH1F object at 0x60fbbddd0090>, 'Ch14_BG_a': <cppyy.gbl.TH1F object at 0x60fbbd170fa0>, 'Ch15_BG_a': <cppyy.gbl.TH1F object at 0x60fbc36502a0>, 'Ch16_BG_a': <cppyy.gbl.TH1F object at 0x60fbc3796000>, 'Siphra_sum_BG_a': <cppyy.gbl.TH1F object at 0x60fbc39f6310>, 'Manual_sum_BG_a': <cppyy.gbl.TH1F object at 0x60f

In [7]:
channels = ["Siphra_sum", "Manual_sum"]


if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1400, 1000)
cv.Divide(2, 2)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for i, source in enumerate(sources):
    cv.cd(i+1)
    cv.cd(i+1).SetLogy()
    lg = lgds[i]
    for j, channel in enumerate(channels):
        hist = hists[f"{channel}_{source}"]
        lg.AddEntry(hist, channel)
        hist.SetLineColor(colors[j+1])
        if j == 0:
            hist.Draw("hist")
        else:
            hist.Draw("sames hist")
        ROOT.gPad.Update()
        
        
    lg.Draw()
    cv.Draw()

#print(hists)

In [5]:
sources = ['Cs-137', 'Na-22', 'Co-60']
channels = ["Siphra_sum", "Manual_sum"]

energies = {}
energies['Cs-137'] = CS137_MEV
energies['Na-22'] = NA22_MEV
energies['Co-60'] = CO60_MEV

ranges_ss = {}
ranges_ss["Co-60"] = [[470, 640], [900, 1050]]
ranges_ss['Na-22'] = [[230, 350], [500, 630], [670, 770]]
ranges_ss['Cs-137'] = [[230, 420]]

ranges_ms = {}
ranges_ms["Co-60"] = [[440, 590], [820, 920]]
ranges_ms['Na-22'] = [[230, 350], [470, 580], [620, 700]]
ranges_ms['Cs-137'] = [[230, 390]]

ranges = {}
ranges['Siphra_sum'] = ranges_ss
ranges['Manual_sum'] = ranges_ms

energy_vs_diff = []

for source in sources:
    means = []
    for i, channel in enumerate(channels):
        rng = ranges[channel][source]
        hist = hists[f"{channel}_{source}"]
        means.append([])
        
        for r in rng:
            gauss = ROOT.TF1(f"gauss_{channel}_{source}", "gaus", r[0], r[1])
            hist.Fit(gauss, "R+S")
            mean = gauss.GetParameter(1)
            means[i].append(mean)

    for m in range(len(means[0])):
        diff = means[0][m]/means[1][m]
        diff2 = (means[0][m]-means[1][m])/means[0][m]
        #print(diff2)
        print(f"{source} (energy: {energies[source][m]:.4g} MeV), diff: {(diff2)*100:.3g} %") 
        energy_vs_diff.append((energies[source][m], diff2*100))


KeyError: 'Siphra_sum_Cs-137'

In [ ]:
import matplotlib.pyplot as plt

#data = [(1, 2), (2, 4), (3, 3), (4, 7), (5, 6)]

x, y = zip(*energy_vs_diff)

m, b = np.polyfit(x, y, 1)

plt.scatter(x, y, marker="o")

plt.plot(x, np.array(x) * m + b, label=f"fit: y = {m:.2f}x + {b:.2f}")

plt.xlabel("Energy [MeV]")
plt.ylabel("Difference [%]")
plt.show()